In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ward = pd.read_csv("ward_admissions.csv")
ward["heart_rate"] = ward["heart_rate"].replace(999, np.nan)
ward["heart_rate"] = ward["heart_rate"].fillna(
ward["heart_rate"].median())
num_cols = ["age", "heart_rate",
"resp_rate", "length_of_stay"]
ward[num_cols].skew().round(2)

age               0.02
heart_rate        0.84
resp_rate         1.01
length_of_stay    6.21
dtype: float64

In [18]:
ward["los_ew"], ew_edges = pd.cut(
ward["heart_rate"], bins=5, retbins=True)
print(np.round(ew_edges, 2))
ward["los_ew"].value_counts(sort=False)

[ 31.86  60.8   89.6  118.4  147.2  176.  ]


los_ew
(31.856, 60.8]     47
(60.8, 89.6]      664
(89.6, 118.4]     468
(118.4, 147.2]     11
(147.2, 176.0]     10
Name: count, dtype: int64

In [19]:
labels = ["Bin 1", "Bin 2", "Bin 3", "Bin 4", "Bin 5"]
ward["los_ew_lab"] = pd.cut(ward["heart_rate"],
bins=5, labels=labels)
ward.groupby("los_ew", observed=True)[
  "deteriorated"].agg(["count", "mean"])

,count,mean
los_ew,,
"(31.856, 60.8]",47,0.021277
"(60.8, 89.6]",664,0.096386
"(89.6, 118.4]",468,0.224359
"(118.4, 147.2]",11,1.000000
"(147.2, 176.0]",10,1.000000


In [20]:
ward["age_ew"] = pd.cut(ward["age"], bins=5)
ward.groupby("age_ew", observed=True)[
"deteriorated"].agg(["count", "mean"])

,count,mean
age_ew,,
"(21.923, 37.4]",20,0.200000
"(37.4, 52.8]",172,0.087209
"(52.8, 68.2]",494,0.111336
"(68.2, 83.6]",383,0.195822
"(83.6, 99.0]",131,0.320611


K- mean on heart_rate, using supervised to get the bins

In [27]:
from sklearn.preprocessing import KBinsDiscretizer
kb = KBinsDiscretizer(n_bins=4, encode="ordinal",
                      strategy="kmeans")
ward["hr_km"] = kb.fit_transform(ward[["heart_rate"]]).ravel()


c:\Users\rotic\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


In [28]:

print(np.round(kb.bin_edges_[0], 3))

[ 32.     76.984  94.468 132.472 176.   ]


In [29]:
for s in ["uniform", "quantile", "kmeans"]:
     k = KBinsDiscretizer(n_bins=4, encode="ordinal",
     strategy=s)
     k.fit(ward[["heart_rate"]])
     print(s, np.round(k.bin_edges_[0], 1))

uniform [ 32.  68. 104. 140. 176.]
quantile [ 32.  76.  86.  95. 176.]
kmeans [ 32.   77.   94.5 132.5 176. ]


c:\Users\rotic\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

train, test = train_test_split(ward, test_size=0.25,
    random_state=42, stratify=ward["deteriorated"])
tree = DecisionTreeClassifier(max_depth=2,
    min_samples_leaf=50, random_state=42)

tree.fit(train[["age"]], train["deteriorated"])
print(export_text(tree, feature_names=["age"]))

|--- age <= 70.50
|   |--- age <= 62.50
|   |   |--- class: 0
|   |--- age >  62.50
|   |   |--- class: 0
|--- age >  70.50
|   |--- age <= 84.50
|   |   |--- class: 0
|   |--- age >  84.50
|   |   |--- class: 0



In [31]:
is_split = tree.tree_.feature != -2 # -2 marks a leaf
th = sorted(tree.tree_.threshold[is_split])
edges = [-np.inf] + th + [np.inf]
train["age_tree"] = pd.cut(train["age"], bins=edges)
test["age_tree"] = pd.cut(test["age"], bins=edges)